In [1]:
import numpy as np
import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler
from sobol_seq import i4_sobol_generate

def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X, y

# Load and shuffle dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
# Set dimensions for hyperdimensional space
D = 10000
NUM_CLASSES = 2
NUM_SAMPLES = X_train.shape[0]

# Create Sobol sequence for random projection matrix
print("Generating Sobol sequence...")
sobol_sequence = i4_sobol_generate(X_train.shape[1], D)

def project_data(data, seq):
    return np.dot(data, seq.T)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = project_data(X_train, sobol_sequence)

# Create class hypervectors by summing the projected vectors of each class
class_hypervectors = np.zeros((NUM_CLASSES, D))
for i in range(NUM_SAMPLES):
    class_hypervectors[y_train[i]] += X_train_proj[i]

# Normalize the class hypervectors
class_hypervectors = class_hypervectors / np.linalg.norm(class_hypervectors, axis=1, keepdims=True)

def classify(images, class_hypervectors):
    similarities = cosine_similarity(images, class_hypervectors)
    classifications = np.argmax(similarities, axis=1)
    return classifications
    
end_train_time = time.time()
training_time = end_train_time - start_train_time
# Training accuracy
predictions_train = classify(X_train_proj, class_hypervectors)
acc_train = accuracy_score(y_train, predictions_train) * 100
print("Convensional HDC + Cosine Similarity Train Accuracy: ", acc_train)

# Measure inference time
start_inference_time = time.time()

# Project test data to hyperdimensional space
X_test_proj = project_data(X_test, sobol_sequence)

# Classify test data
predictions_test = classify(X_test_proj, class_hypervectors)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

acc_test = accuracy_score(y_test, predictions_test) * 100
print("Convensional HDC + Cosine Similarity Test Accuracy: ", acc_test)

# Calculate model memory requirement
model_memory = sobol_sequence.nbytes + class_hypervectors.nbytes

print("Convensional HDC + Cosine Similarity Training Time: {:.4f} seconds".format(training_time))
print("Convensional HDC + Cosine Similarity Inference Time: {:.4f} seconds".format(inference_time))
print("Convensional HDC + Cosine Similarity Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating Sobol sequence...
Convensional HDC + Cosine Similarity Train Accuracy:  91.0761154855643
Convensional HDC + Cosine Similarity Test Accuracy:  89.36170212765957
Convensional HDC + Cosine Similarity Training Time: 0.0270 seconds
Convensional HDC + Cosine Similarity Inference Time: 0.0299 seconds
Convensional HDC + Cosine Similarity Model Memory Required: 2.4414 MB


In [2]:
import numpy as np
import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.random_projection import GaussianRandomProjection

def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X, y

# Load and shuffle dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Set dimensions for hyperdimensional space
D = 10000
NUM_CLASSES = 2
NUM_SAMPLES = X_train.shape[0]

# Generate random projection matrix
print("Generating Random Projection matrix...")
projection_matrix = GaussianRandomProjection(n_components=D).fit(X_train)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = projection_matrix.transform(X_train)

# Create class hypervectors by summing the projected vectors of each class
class_hypervectors = np.zeros((NUM_CLASSES, D))
for i in range(NUM_SAMPLES):
    class_hypervectors[y_train[i]] += X_train_proj[i]

# Normalize the class hypervectors
class_hypervectors = class_hypervectors / np.linalg.norm(class_hypervectors, axis=1, keepdims=True)

def classify(images, class_hypervectors):
    similarities = cosine_similarity(images, class_hypervectors)
    classifications = np.argmax(similarities, axis=1)
    return classifications
    
end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = classify(X_train_proj, class_hypervectors)
acc_train = accuracy_score(y_train, predictions_train) * 100
print("Convensional HDC + Cosine Similarity Train Accuracy: ", acc_train)

# Measure inference time
start_inference_time = time.time()

# Project test data to hyperdimensional space
X_test_proj = projection_matrix.transform(X_test)

# Classify test data
predictions_test = classify(X_test_proj, class_hypervectors)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

acc_test = accuracy_score(y_test, predictions_test) * 100
print("Convensional HDC + Cosine Similarity Test Accuracy: ", acc_test)

# Calculate model memory requirement
model_memory = projection_matrix.components_.nbytes + class_hypervectors.nbytes

print("Convensional HDC + Cosine Similarity Training Time: {:.4f} seconds".format(training_time))
print("Convensional HDC + Cosine Similarity Inference Time: {:.4f} seconds".format(inference_time))
print("Convensional HDC + Cosine Similarity Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating Random Projection matrix...
Convensional HDC + Cosine Similarity Train Accuracy:  90.81364829396325
Convensional HDC + Cosine Similarity Test Accuracy:  89.36170212765957
Convensional HDC + Cosine Similarity Training Time: 0.0333 seconds
Convensional HDC + Cosine Similarity Inference Time: 0.0223 seconds
Convensional HDC + Cosine Similarity Model Memory Required: 2.4414 MB


C:\Users\abuka\anaconda3\Lib\site-packages\sklearn\random_projection.py:398: DataDimensionalityWarning: The number of components is higher than the number of features: n_features < n_components (30 < 10000).The dimensionality of the problem will not be reduced.
  warnings.warn(


In [6]:
import numpy as np
import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score
from sklearn.kernel_approximation import SkewedChi2Sampler

def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X, y

# Load and shuffle dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Set dimensions for hyperdimensional space
D = 100
NUM_CLASSES = 2
NUM_SAMPLES = X_train.shape[0]

# Generate Sparse Random Features
print("Generating Sparse Random Features...")
srf = SkewedChi2Sampler(n_components=D, random_state=42)
X_train_proj = srf.fit_transform(X_train)

# Measure training time
start_train_time = time.time()

# Create class hypervectors by summing the projected vectors of each class
class_hypervectors = np.zeros((NUM_CLASSES, D))
for i in range(NUM_SAMPLES):
    class_hypervectors[y_train[i]] += X_train_proj[i]

# Normalize the class hypervectors
class_hypervectors = class_hypervectors / np.linalg.norm(class_hypervectors, axis=1, keepdims=True)

def classify(images, class_hypervectors):
    similarities = cosine_similarity(images, class_hypervectors)
    classifications = np.argmax(similarities, axis=1)
    return classifications
    
end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = classify(X_train_proj, class_hypervectors)
acc_train = accuracy_score(y_train, predictions_train) * 100
print("Convensional HDC + Cosine Similarity Train Accuracy: ", acc_train)

# Measure inference time
start_inference_time = time.time()

# Project test data to hyperdimensional space
X_test_proj = srf.transform(X_test)

# Classify test data
predictions_test = classify(X_test_proj, class_hypervectors)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

acc_test = accuracy_score(y_test, predictions_test) * 100
print("Convensional HDC + Cosine Similarity Test Accuracy: ", acc_test)

# Calculate model memory requirement
model_memory = X_train_proj.nbytes + class_hypervectors.nbytes

print("Convensional HDC + Cosine Similarity Training Time: {:.4f} seconds".format(training_time))
print("Convensional HDC + Cosine Similarity Inference Time: {:.4f} seconds".format(inference_time))
print("Convensional HDC + Cosine Similarity Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating Sparse Random Features...
Convensional HDC + Cosine Similarity Train Accuracy:  89.23884514435696
Convensional HDC + Cosine Similarity Test Accuracy:  92.5531914893617
Convensional HDC + Cosine Similarity Training Time: 0.0024 seconds
Convensional HDC + Cosine Similarity Inference Time: 0.0000 seconds
Convensional HDC + Cosine Similarity Model Memory Required: 0.2922 MB


In [3]:
X_train_proj.shape

(381, 8)

In [33]:
import numpy as np
import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.kernel_approximation import SkewedChi2Sampler
from sklearn.svm import SVC

def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

def load_dataset():
    breast_cancer = load_breast_cancer()
    X, y = breast_cancer.data, breast_cancer.target
    return X, y

# Load and shuffle dataset
X, y = load_dataset()
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Set dimensions for hyperdimensional space
D = 100

# Generate Sparse Random Features
print("Generating Sparse Random Features...")
srf = SkewedChi2Sampler(n_components=D, random_state=42)
X_train_proj = srf.fit_transform(X_train)

# Measure training time
start_train_time = time.time()

# Train SVM classifier
svm = SVC(kernel='linear')
svm.fit(X_train_proj, y_train)

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Predict training labels
predictions_train = svm.predict(X_train_proj)

# Calculate training accuracy
acc_train = accuracy_score(y_train, predictions_train) * 100
print("SVM Training Accuracy: ", acc_train)

# Measure inference time
start_inference_time = time.time()

# Project test data to hyperdimensional space
X_test_proj = srf.transform(X_test)

# Predict test labels
predictions_test = svm.predict(X_test_proj)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Calculate test accuracy
acc_test = accuracy_score(y_test, predictions_test) * 100
print("SVM Test Accuracy: ", acc_test)

# Calculate model memory requirement
model_memory = X_train_proj.nbytes + svm.support_vectors_.nbytes

print("SVM Training Time: {:.4f} seconds".format(training_time))
print("SVM Inference Time: {:.4f} seconds".format(inference_time))
print("SVM Model Memory Required: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating Sparse Random Features...
SVM Training Accuracy:  95.01312335958005
SVM Test Accuracy:  93.08510638297872
SVM Training Time: 0.0064 seconds
SVM Inference Time: 0.0025 seconds
SVM Model Memory Required: 0.3662 MB


In [42]:
from sklearn.inspection import permutation_importance
breast_cancer = load_breast_cancer()

print("Analysing Feature Importance...")
feature_importances = permutation_importance(svm, X_train_proj, y_train, scoring='accuracy')
feature_importances


Analysing Feature Importance...


{'importances_mean': array([ 7.87401575e-03,  2.09973753e-03,  5.24934383e-04,  5.24934383e-04,
         7.87401575e-03,  1.57480315e-03,  3.14960630e-03,  1.04986877e-03,
         5.24934383e-03,  1.57480315e-03,  2.62467192e-03,  1.57480315e-03,
         1.04986877e-03,  2.09973753e-03,  2.09973753e-03,  1.04986877e-03,
         2.09973753e-03,  2.09973753e-03,  1.57480315e-03,  4.19947507e-03,
         4.72440945e-03,  5.24934383e-04,  1.57480315e-03,  2.09973753e-03,
         5.24934383e-04,  3.14960630e-03,  0.00000000e+00,  1.04986877e-03,
         9.44881890e-03,  5.24934383e-04,  0.00000000e+00,  1.57480315e-03,
        -5.24934383e-04,  2.09973753e-03,  2.22044605e-17,  6.29921260e-03,
         0.00000000e+00,  3.14960630e-03,  1.57480315e-03,  1.04986877e-03,
         2.62467192e-03,  1.04986877e-03,  0.00000000e+00,  1.57480315e-03,
         1.57480315e-03,  5.24934383e-04,  1.04986877e-03,  3.14960630e-03,
         1.04986877e-03,  4.72440945e-03,  3.14960630e-03,  1.049868

In [43]:
# Print feature importances (sort by importance for readability)
for feature, importance in sorted(zip(breast_cancer.feature_names, feature_importances), key=lambda x: x[1], reverse=True):
    print("Feature: ",feature, "Importance", importance)

Feature:  mean texture Importance importances_std
Feature:  mean radius Importance importances_mean
Feature:  mean perimeter Importance importances


In [29]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained("gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

C:\Users\abuka\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abuka\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [30]:
import torch
from archai.quantization.ptq import dynamic_quantization_torch

torch.set_num_threads(1)
model_qnt = dynamic_quantization_torch(model)

2024-06-04 14:17:54,407 - archai.quantization.ptq — INFO —  Quantizing model ...


In [31]:
model_qnt

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): QuantizedEmbedding(num_embeddings=50257, embedding_dim=768, dtype=torch.quint8, qscheme=torch.per_channel_affine_float_qparams)
    (wpe): QuantizedEmbedding(num_embeddings=1024, embedding_dim=768, dtype=torch.quint8, qscheme=torch.per_channel_affine_float_qparams)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, 